In [12]:
import pandas as pd
from glob import glob
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt


import torch
from torch.utils.data import Dataset , DataLoader

import torchvision.transforms as T

import torch.nn as nn

import torch.nn.functional as F

#import timm
import torchvision.models  as models
import transformers 
import torch.optim as optim
from transformers  import AutoImageProcessor  ,AutoModel


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class Config:
    
    data_dir="/kaggle/input/csiro-biomass"
    
    train_csv=pd.read_csv(os.path.join(data_dir,"train.csv"))
    test_csv=os.path.join(data_dir,"test.csv")
    train_images=os.path.join(data_dir,"train")
    test_images=os.path.join(data_dir,"test")
    batch=12
    lr=1e-3
    epochs=25
    n_folds = 5


targets_configs = ["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g"] 
targets = ["Dry_Clover_g","Dry_Dead_g","Dry_Green_g","Dry_Total_g","GDM_g"]

# DataSet

In [17]:
train_wide = Config.train_csv.pivot_table(
    index=["image_path","Sampling_Date","State","Species","Pre_GSHH_NDVI","Height_Ave_cm"],
    columns="target_name",
    values="target",
    aggfunc="first"
).reset_index()

train_wide["full_path"] = train_wide["image_path"].apply(
    lambda x: os.path.join(Config.data_dir, x)
)#****

image_paths = train_wide["full_path"].values
target_cols_3 = ["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g"]
y3 = train_wide[target_cols_3].values.astype("float32")

train_wide = Config.train_csv.pivot_table(
    index=['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm'],
    columns='target_name',
    values='target',
    aggfunc='first'
).reset_index()


class Biomass(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths, self.labels, self.transform = paths, labels, transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(self.paths[idx]), cv2.COLOR_BGR2RGB)
        img = self.transform(img) if self.transform else img
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label



train_transform = T.Compose([
    T.ToPILImage(), T.Resize((224,224)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.ColorJitter(0.1,0.1,0.1,0.05),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

valid_transform = T.Compose([
    T.ToPILImage(), T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

dataset = Biomass(image_paths, y3, transform=train_transform)
train_loader = DataLoader(dataset, batch_size=Config.batch, shuffle=True)



# Functions

In [19]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = model(imgs)
        optimizer.zero_grad()
   
        loss = criterion(preds, labels)

        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

    return running_loss / len(loader.dataset)

class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None

    def step(self, score):
        if self.best_score is None or score < self.best_score - self.min_delta:
            self.best_score = score
            self.counter = 0
            return True
        else:
            self.counter += 1
            return False

    def should_stop(self):
        return self.counter >= self.patience

def weighted_r2_score(y_true, y_pred):
        weights = np.array([0.1, 0.1, 0.1, 0.2, 0.5])
        r2s = []
    
        for i in range(5):
            yt, yp = y_true[:, i], y_pred[:, i]
            ss_res = np.sum((yt - yp) ** 2)
            ss_tot = np.sum((yt - yt.mean()) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
            r2s.append(r2)
    
        r2s = np.array(r2s)
        return np.sum(r2s * weights) / weights.sum(), r2s

def expand_predictions_torch(preds_3, targets):
    """
    [torch] Expand the 3 NN predictions (N, 3) to 5 predictions (N, 5).
    """
    if targets == ["Dry_Green_g", "Dry_Total_g", "GDM_g"]:
        # clip
        P_Green = torch.clamp(preds_3[:, 0], min=0)
        P_Total = torch.clamp(preds_3[:, 1], min=0)
        P_GDM = torch.clamp(preds_3[:, 2], min=0)
        
        # Compute derived targets based on constraints.
        P_Clover = torch.clamp(P_GDM - P_Green, min=0)
        P_Dead = torch.clamp(P_Total - P_GDM, min=0)
    
    elif targets == ["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g"]:
        P_Clover = torch.clamp(preds_3[:, 0], min=0)
        P_Dead = torch.clamp(preds_3[:, 1], min=0)
        P_Green = torch.clamp(preds_3[:, 2], min=0)

        # Compute derived targets based on constraints.
        P_GDM = torch.clamp(P_Green + P_Clover, min=0)
        P_Total = torch.clamp(P_GDM + P_Dead, min=0)
    elif targets == ["Dry_Clover_g", "Dry_Dead_g", "GDM_g"]:
        P_Clover = torch.clamp(preds_3[:, 0], min=0)
        P_Dead =torch.clamp(preds_3[:, 1], min=0)
        P_GDM = torch.clamp(preds_3[:, 2], min=0)

        # Compute derived targets based on constraints.
        P_Green = torch.clamp(P_GDM - P_Clover, min=0)
        P_Total = torch.clamp(P_GDM + P_Dead, min=0)

    
    preds_5 = torch.stack(
        [
            P_Clover, # Index 0
            P_Dead,   # Index 1
            P_Green,  # Index 2
            P_Total,  # Index 3
            P_GDM     # Index 4
        ],
        dim=1
    )
    return preds_5


def val_fn(model, dataloader, criterion, device, targets):
    """
    Validation / evaluation for 1 epoch.
    Assumes the dataset returns only 3 independent targets per sample.
    """
    model.eval()
    total_loss = 0
    all_targets_5_np = []
    all_preds_5_np = []
    
    with torch.no_grad():
        for img, targets_3 in dataloader:
            img = img.to(device)
            targets_3 = targets_3.to(device)
    
            outputs_3 = model(img)                  # (B, 3)
            loss = criterion(outputs_3, targets_3)  # loss on 3 only
            total_loss += loss.item() * img.size(0)
    
            preds_5 = expand_predictions_torch(outputs_3, targets)
            targs_5 = expand_predictions_torch(targets_3, targets)
    
            all_preds_5_np.append(preds_5.cpu().numpy())
            all_targets_5_np.append(targs_5.cpu().numpy())


    # Average loss over dataset

    
    val_loss = total_loss / len(dataloader.dataset)

    # Concatenate all predictions / targets
    y_true_5 = np.concatenate(all_targets_5_np, axis=0)
    y_pred_5 = np.concatenate(all_preds_5_np, axis=0)

    # Compute weighted R² metric on full 5 targets
    weighted_r2, _ = weighted_r2_score(y_true_5, y_pred_5)

    return val_loss, weighted_r2

# Model Loading 

In [20]:
import torch
import torch.nn as nn
from transformers import AutoModel

class DINOv2(nn.Module):
    def __init__(self, n_targets=3, freeze_backbone=True, hidden_dim=512, dropout=0.01):
        super().__init__()

        BACKBONE_ID = "/kaggle/input/dinov2/pytorch/large/1"
        self.backbone = AutoModel.from_pretrained(BACKBONE_ID, local_files_only=True)

        embed_dim = self.backbone.config.hidden_size   # ✅ fix here
        pooled_dim = embed_dim * 2

        self.regressor = nn.Sequential(
            nn.Linear(pooled_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_targets)
        )

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def _pool_feats(self, outputs):
        feat = outputs.last_hidden_state[:, 1:] 
        avg_pool = feat.mean(dim=1)
        max_pool = feat.amax(dim=1) # Because the output is (BATCH, Token ,dim )   so we want to avg of  Token only 
        return torch.cat([avg_pool, max_pool], dim=1)

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        emb = self._pool_feats(out)
        return self.regressor(emb)


# Cross Fold

In [ ]:
from sklearn.model_selection import KFold
import numpy as np
import torch

# Store OOF predictions in full 5-target format
oof_5 = np.zeros((len(train_wide), 5), dtype=np.float32)

kf = KFold(n_splits=Config.n_folds, shuffle=True, random_state=42)

for fold, (tr_idx, va_idx) in enumerate(kf.split(train_wide)):
    print(f"\n======== FOLD {fold+1}/{Config.n_folds} ========")

    targets = target_cols_3

    # split arrays
    tr_paths = image_paths[tr_idx]
    va_paths = image_paths[va_idx]
    tr_y3 = y3[tr_idx]
    va_y3 = y3[va_idx]

    # datasets & loaders
    train_dataset = Biomass(tr_paths, tr_y3, transform=train_transform)
    val_dataset   = Biomass(va_paths, va_y3, transform=valid_transform)

    train_loader = DataLoader(train_dataset, batch_size=Config.batch, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=Config.batch, shuffle=False)

    # model
    model = DINOv2(n_targets=3, freeze_backbone=True).to(device)
    optimizer = torch.optim.AdamW(model.regressor.parameters(), lr=Config.lr, weight_decay=1e-3)
    criterion = nn.SmoothL1Loss(beta=0.5)

    best_r2 = -np.inf

    # ===========================
    # Train
    # ===========================
    for epoch in range(Config.epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, weighted_r2 = val_fn(model, val_loader, criterion, device, targets)

        print(
            f"Epoch {epoch+1}/{Config.epochs} | "
            f"Train {train_loss:.4f} | Val {val_loss:.4f} | R2 {weighted_r2:.4f}"
        )

        # ✅ save best by competition metric
        if weighted_r2 > best_r2:
            best_r2 = weighted_r2
            torch.save(model.state_dict(), f"model_fold_{fold}.pth")

    # ===========================
    # OOF prediction (best checkpoint)
    # ===========================
    model.load_state_dict(torch.load(f"model_fold_{fold}.pth", map_location=device))
    model.eval()

    fold_preds_3 = []
    with torch.no_grad():
        for imgs, _ in val_loader:
            imgs = imgs.to(device)
            preds_3 = model(imgs)   # (B, 3)
            fold_preds_3.append(preds_3.cpu())

    fold_preds_3 = torch.cat(fold_preds_3, dim=0)  # (Nval, 3)
    fold_preds_5 = expand_predictions_torch(fold_preds_3, target_cols_3)  # (Nval, 5)

    oof_5[va_idx] = fold_preds_5.numpy()

# ===========================
# Final OOF evaluation
# ===========================
y_true_5 = expand_predictions_torch(torch.tensor(y3), target_cols_3).numpy()
final_r2, per_target_r2 = weighted_r2_score(y_true_5, oof_5)

print("\n======== FINAL OOF RESULTS ========")
print("Final OOF Weighted R2:", final_r2)
print("Per-target R2:", per_target_r2)



======== FOLD 1/3 ========
